# 🧪 FatTummy — Full Test Suite
Runs **unit + integration tests** for every module.
Each cell is independent. `PASS ✅` / `FAIL ❌` is printed per test.

In [ ]:
import sys, os, traceback
sys.path.insert(0, os.path.abspath('.'))

PASS = '\033[92mPASS ✅\033[0m'
FAIL = '\033[91mFAIL ❌\033[0m'
_results = []

def run_test(name, fn):
    try:
        fn()
        print(f'{PASS}  {name}')
        _results.append((name, True, None))
    except Exception as e:
        print(f'{FAIL}  {name}')
        traceback.print_exc()
        _results.append((name, False, e))

print('Setup complete. sys.path has Fat Pack root.')

---
## 1. Import & Version

In [ ]:
def test_import():
    import FatTummy as ft
    assert hasattr(ft, '__version__'), 'missing __version__'
    assert isinstance(ft.__version__, str), '__version__ must be a string'
    print(f'  FatTummy version: {ft.__version__}')

def test_public_api_present():
    import FatTummy as ft
    for name in ['build','modelbuild','type','data','temp','chat','generate','finetune','engine','key','hf_login','quantize']:
        assert callable(getattr(ft, name, None)), f'ft.{name} not callable'
    print('  All public API symbols present and callable.')

def test_mooe_accessible():
    import FatTummy as ft
    MOOE = ft.MOOE
    assert MOOE is not None
    print(f'  ft.MOOE -> {MOOE}')

run_test('Import FatTummy', test_import)
run_test('Public API symbols', test_public_api_present)
run_test('ft.MOOE accessible', test_mooe_accessible)

---
## 2. Exception Hierarchy

In [ ]:
from FatTummy.exceptions import (
    FatTummyBaseException, FatTummyConfigurationError, FatTummyDatasetError,
    FatTummyAuthenticationError, FatTummyDependencyError,
    FatTummyUnsupportedBackendError, FatTummyOOMError,
    FatTummyDriverError, FatTummyNetworkError
)

def test_exception_hierarchy():
    for cls in [
        FatTummyConfigurationError, FatTummyDatasetError,
        FatTummyAuthenticationError, FatTummyDependencyError,
        FatTummyUnsupportedBackendError, FatTummyOOMError,
        FatTummyDriverError, FatTummyNetworkError
    ]:
        assert issubclass(cls, FatTummyBaseException), f'{cls} not a subclass of base'
    print('  All exceptions inherit from FatTummyBaseException.')

def test_oom_error_message():
    err = FatTummyOOMError('mock oom')
    assert 'memory' in str(err).lower(), 'OOM message should mention memory'
    assert 'mock oom' in str(err), 'original error should appear in message'
    print(f'  OOMError msg: {err}')

def test_driver_error_gpu():
    err = FatTummyDriverError('gpu', 'driver missing')
    assert 'CUDA' in str(err) or 'cuda' in str(err).lower(), 'GPU driver error should mention CUDA'
    print(f'  DriverError (gpu): {err}')

def test_driver_error_tpu():
    err = FatTummyDriverError('tpu')
    assert 'torch_xla' in str(err) or 'tpu' in str(err).lower()
    print(f'  DriverError (tpu): {err}')

def test_network_error():
    err = FatTummyNetworkError('test_op', 'path/to/cache', 'connection refused')
    msg = str(err)
    assert 'test_op' in msg
    assert 'path/to/cache' in msg
    assert 'connection refused' in msg
    print(f'  NetworkError msg: {err}')

run_test('Exception hierarchy', test_exception_hierarchy)
run_test('OOMError message', test_oom_error_message)
run_test('DriverError (gpu)', test_driver_error_gpu)
run_test('DriverError (tpu)', test_driver_error_tpu)
run_test('NetworkError', test_network_error)

---
## 3. Installer / Dependency Audit

In [ ]:
from FatTummy.installer import (
    is_package_installed, missing_dependencies, format_install_hint, ensure_installed
)

def test_is_package_installed_stdlib():
    assert is_package_installed('sys'), 'sys must always be found'
    assert is_package_installed('os'), 'os must always be found'
    assert not is_package_installed('this_package_does_not_exist_xyz123')
    print('  is_package_installed works correctly.')

def test_missing_dependencies_unknown_extra():
    # unknown extra should return empty dict (no registered deps)
    result = missing_dependencies(['nonexistent_extra'])
    assert result == {}, f'expected empty dict, got {result}'
    print('  missing_dependencies handles unknown extras gracefully.')

def test_format_install_hint_no_missing():
    # 'ollama' extra has no required packages
    hint = format_install_hint(['ollama'])
    assert hint == '', f'expected empty hint, got: {hint!r}'
    print('  format_install_hint returns empty string when nothing is missing.')

def test_ensure_installed_runs():
    # Should not raise — just prints hints
    ensure_installed(api_only=True)
    ensure_installed(api_only=False)
    print('  ensure_installed() completed without exception.')

run_test('is_package_installed (stdlib)', test_is_package_installed_stdlib)
run_test('missing_dependencies (unknown extra)', test_missing_dependencies_unknown_extra)
run_test('format_install_hint (no missing)', test_format_install_hint_no_missing)
run_test('ensure_installed()', test_ensure_installed_runs)

---
## 4. FatTummyEngine — Builder API

In [ ]:
from FatTummy.engine import FatTummyEngine
from FatTummy.exceptions import (
    FatTummyConfigurationError, FatTummyUnsupportedBackendError, FatTummyAuthenticationError
)

def test_engine_fluent_chain():
    e = FatTummyEngine()
    result = e.modelbuild('tiny').temp(0.5).token_limit(256).epochs(2)
    assert result is e, 'fluent chain must return self'
    assert e._param == 'tiny'
    assert e._temperature == 0.5
    assert e._token_limit == 256
    assert e._epochs == 2
    print('  Fluent chain returns self and sets state.')

def test_engine_bad_temp():
    e = FatTummyEngine()
    try:
        e.temp(0)
        assert False, 'should have raised'
    except FatTummyConfigurationError as ex:
        print(f'  Correct error for temp=0: {ex}')

def test_engine_bad_token_limit():
    e = FatTummyEngine()
    try:
        e.token_limit(-1)
        assert False, 'should have raised'
    except FatTummyConfigurationError as ex:
        print(f'  Correct error for token_limit=-1: {ex}')

def test_engine_bad_epochs():
    e = FatTummyEngine()
    try:
        e.epochs(0)
        assert False, 'should have raised'
    except FatTummyConfigurationError as ex:
        print(f'  Correct error for epochs=0: {ex}')

def test_engine_unknown_engine():
    e = FatTummyEngine()
    try:
        e.engine('fakellm')
        assert False, 'should have raised'
    except FatTummyUnsupportedBackendError as ex:
        print(f'  Correct error for unknown engine: {ex}')

def test_engine_aliases():
    e = FatTummyEngine()
    e.engine('huggingface')
    assert e._engine_name == 'hf'
    e.engine('claude')
    assert e._engine_name == 'anthropic'
    e.engine('google')
    assert e._engine_name == 'gemini'
    print('  Engine aliases resolve correctly.')

def test_engine_action_valid():
    e = FatTummyEngine()
    for a in ['make', 'finetune', 'api', 'chat']:
        e.action(a)
        assert e._action == a
    print('  All valid actions accepted.')

def test_engine_action_invalid():
    e = FatTummyEngine()
    try:
        e.action('dance')
        assert False, 'should have raised'
    except FatTummyConfigurationError as ex:
        print(f'  Correct error for invalid action: {ex}')

def test_engine_data_extends():
    e = FatTummyEngine()
    e.data('source_a', 'source_b')
    e.data('source_c')
    assert e._data_sources == ['source_a', 'source_b', 'source_c']
    print('  data() accumulates sources correctly.')

def test_engine_key_stored():
    e = FatTummyEngine()
    e.key('sk-test123')
    assert e._api_key == 'sk-test123'
    print('  key() stores API key.')

def test_engine_quantize_stored():
    e = FatTummyEngine()
    e.quantize('4bit')
    assert e._quantization == '4bit'
    print('  quantize() stores mode.')

def test_engine_generate_no_key_cloud():
    e = FatTummyEngine()
    e._engine_name = 'openai'
    e._compiled = True
    try:
        e.generate('hello')
        assert False, 'should have raised FatTummyAuthenticationError'
    except FatTummyAuthenticationError as ex:
        print(f'  Correct auth error with no key: {ex}')

run_test('Fluent chain + state', test_engine_fluent_chain)
run_test('Bad temperature', test_engine_bad_temp)
run_test('Bad token_limit', test_engine_bad_token_limit)
run_test('Bad epochs', test_engine_bad_epochs)
run_test('Unknown engine raises', test_engine_unknown_engine)
run_test('Engine aliases', test_engine_aliases)
run_test('Valid actions', test_engine_action_valid)
run_test('Invalid action raises', test_engine_action_invalid)
run_test('data() accumulates', test_engine_data_extends)
run_test('key() stores', test_engine_key_stored)
run_test('quantize() stores', test_engine_quantize_stored)
run_test('generate() no key → AuthError', test_engine_generate_no_key_cloud)

---
## 5. MOOE Native Model

In [ ]:
try:
    import torch
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False
    print('WARNING: PyTorch not installed — MOOE tests will be skipped.')

if _HAS_TORCH:
    from FatTummy.models.mooe import MOOE, MOOEConfig, MOOELayer, Expert

    def test_mooe_config_defaults():
        cfg = MOOEConfig()
        assert cfg.hidden_size == 256
        assert cfg.num_layers == 4
        assert cfg.vocab_size == 32000
        assert cfg.num_hidden_layers == cfg.num_layers  # property alias
        assert cfg.num_experts_per_tok == cfg.top_k     # property alias
        print('  MOOEConfig defaults and aliases OK.')

    def test_mooe_config_invalid():
        try:
            MOOEConfig(hidden_size=0)
            assert False, 'should have raised'
        except ValueError as e:
            print(f'  Correct ValueError for hidden_size=0: {e}')

    def test_mooe_config_topk_invalid():
        try:
            MOOEConfig(top_k=10, num_experts=4)
            assert False, 'should have raised'
        except ValueError as e:
            print(f'  Correct ValueError for top_k>num_experts: {e}')

    def test_mooe_forward_tiny():
        cfg = MOOEConfig(hidden_size=32, intermediate_size=64, num_layers=1, num_experts=2, top_k=1, vocab_size=64)
        model = MOOE(cfg)
        ids = torch.randint(0, 64, (1, 8))
        out = model(ids)
        assert 'logits' in out, 'must have logits'
        assert 'aux_loss' in out, 'must have aux_loss'
        assert out['logits'].shape == (1, 8, 64), f'bad logits shape: {out["logits"].shape}'
        print(f'  MOOE forward tiny OK. logits shape: {out["logits"].shape}')

    def test_mooe_forward_with_labels():
        cfg = MOOEConfig(hidden_size=32, intermediate_size=64, num_layers=1, num_experts=2, top_k=1, vocab_size=64)
        model = MOOE(cfg)
        ids = torch.randint(0, 64, (2, 10))
        labels = ids.clone()
        out = model(ids, labels=labels)
        assert 'loss' in out, 'labels should produce a loss key'
        assert out['loss'].item() > 0, 'loss should be positive'
        print(f'  MOOE forward with labels: loss={out["loss"].item():.4f}')

    def test_mooe_generate():
        cfg = MOOEConfig(hidden_size=32, intermediate_size=64, num_layers=1, num_experts=2, top_k=1, vocab_size=64)
        model = MOOE(cfg)
        ids = torch.tensor([[1, 2, 3]])
        out = model.generate(ids, max_new_tokens=5)
        assert out.shape == (1, 8), f'generated shape should be (1, 8), got {out.shape}'
        print(f'  MOOE.generate OK: output shape {out.shape}')

    def test_mooe_class_via_engine():
        """Passing the MOOE class (not string) to engine.type() should build a native model."""
        from FatTummy.engine import FatTummyEngine
        import FatTummy as ft
        e = FatTummyEngine()
        e.modelbuild('tiny')
        e.type(ft.MOOE)  # Pass class, not string
        assert e._model_instance is not None, 'model_instance should be set'
        print(f'  MOOE class passed to type() created instance: {type(e._model_instance).__name__}')

    def test_mooe_string_via_engine():
        """Passing 'mooe' string to engine.type() should also build a native model."""
        from FatTummy.engine import FatTummyEngine
        e = FatTummyEngine()
        e.modelbuild('small')
        e.type('mooe')
        assert e._model_instance is not None
        print(f'  mooe string via type() created instance: {type(e._model_instance).__name__}')

    def test_native_generate_via_engine():
        """Engine._generate_native should produce a string output."""
        from FatTummy.engine import FatTummyEngine
        e = FatTummyEngine()
        e.type('mooe').modelbuild('tiny').token_limit(8)
        out = e.generate('hello')
        assert isinstance(out, str), f'output must be str, got {type(out)}'
        print(f'  Native generate output (first 30 chars): {repr(out[:30])}')

    run_test('MOOEConfig defaults', test_mooe_config_defaults)
    run_test('MOOEConfig invalid hidden_size', test_mooe_config_invalid)
    run_test('MOOEConfig invalid top_k', test_mooe_config_topk_invalid)
    run_test('MOOE forward tiny', test_mooe_forward_tiny)
    run_test('MOOE forward with labels', test_mooe_forward_with_labels)
    run_test('MOOE.generate', test_mooe_generate)
    run_test('MOOE class via engine.type()', test_mooe_class_via_engine)
    run_test('mooe string via engine.type()', test_mooe_string_via_engine)
    run_test('Native generate via engine', test_native_generate_via_engine)
else:
    print('Skipping MOOE tests (PyTorch not installed)')

---
## 6. Native Model Scales (lion / spacebyte)

In [ ]:
if _HAS_TORCH:
    from FatTummy.engine import FatTummyEngine

    def test_lion_engine():
        e = FatTummyEngine()
        e.engine('lion').modelbuild('tiny').type('lion')
        assert e._model_instance is not None
        cfg = e._model_instance.config
        # lion should have fewer experts
        print(f'  Lion model: num_experts={cfg.num_experts}, top_k={cfg.top_k}')
        assert cfg.top_k == 1

    def test_spacebyte_engine():
        e = FatTummyEngine()
        e.engine('spacebyte').modelbuild('tiny').type('spacebyte')
        assert e._model_instance is not None
        cfg = e._model_instance.config
        assert cfg.vocab_size == 256, f'spacebyte vocab should be 256, got {cfg.vocab_size}'
        print(f'  SpaceByte model: vocab_size={cfg.vocab_size}')

    def test_large_scale_warning():
        """Large scale aliases should fall back to 'small' with a warning."""
        import io, sys
        captured = io.StringIO()
        old_stdout = sys.stdout
        sys.stdout = captured
        e = FatTummyEngine()
        e.modelbuild('1b').type('mooe')
        sys.stdout = old_stdout
        output = captured.getvalue()
        assert 'small' in output.lower(), f'Expected fallback message, got: {output}'
        print(f'  Large scale fallback warning printed: {output.strip()}')

    run_test('Lion engine scale', test_lion_engine)
    run_test('SpaceByte engine scale', test_spacebyte_engine)
    run_test('Large scale alias warning', test_large_scale_warning)
else:
    print('Skipping native scale tests (PyTorch not installed)')

---
## 7. Engine Validation Combinations

In [ ]:
from FatTummy.engine import FatTummyEngine
from FatTummy.exceptions import FatTummyUnsupportedBackendError

def test_ollama_with_slash_fails():
    e = FatTummyEngine()
    e._engine_name = 'ollama'
    try:
        e._validate_combination('ollama', 'namespace/model')
        assert False, 'should have raised'
    except FatTummyUnsupportedBackendError as ex:
        print(f'  Correct error for ollama + slash name: {ex}')

def test_hf_with_non_string_fails():
    e = FatTummyEngine()
    try:
        e._validate_combination('hf', 42)
        assert False, 'should have raised'
    except FatTummyUnsupportedBackendError as ex:
        print(f'  Correct error for hf + non-string: {ex}')

def test_cloud_with_non_string_fails():
    e = FatTummyEngine()
    try:
        e._validate_combination('openai', 12345)
        assert False, 'should have raised'
    except FatTummyUnsupportedBackendError as ex:
        print(f'  Correct error for cloud + non-string: {ex}')

def test_cloud_with_none_ok():
    e = FatTummyEngine()
    # None model_type is allowed for cloud (provider selects default)
    e._validate_combination('openai', None)  # should not raise
    print('  Cloud engine with None model_type is accepted.')

run_test('Ollama + slash name fails', test_ollama_with_slash_fails)
run_test('HF + non-string fails', test_hf_with_non_string_fails)
run_test('Cloud + non-string fails', test_cloud_with_non_string_fails)
run_test('Cloud + None model OK', test_cloud_with_none_ok)

---
## 8. Data Loader

In [ ]:
from FatTummy.data.loader import (
    _split_sources, _looks_like_path, _validate_local_file,
    _validate_hf_repo_id, _parse_hf_source
)
from FatTummy.exceptions import FatTummyDatasetError
import tempfile, os

def test_split_sources_string():
    result = _split_sources('a, b, c')
    assert result == ['a', 'b', 'c'], f'got {result}'
    print(f'  _split_sources string: {result}')

def test_split_sources_list():
    result = _split_sources(['x', 'y,z'])
    assert result == ['x', 'y', 'z'], f'got {result}'
    print(f'  _split_sources list: {result}')

def test_split_sources_none():
    result = _split_sources(None)
    assert result == []
    print('  _split_sources None -> []')

def test_split_sources_invalid_type():
    try:
        _split_sources(42)
        assert False
    except FatTummyDatasetError as e:
        print(f'  Correct error for invalid type: {e}')

def test_looks_like_path_txt():
    assert _looks_like_path('my_data.txt')
    assert _looks_like_path('data/file.csv')
    assert _looks_like_path('./relative/path.json')
    assert not _looks_like_path('namespace/repo')
    print('  _looks_like_path correctly identifies paths vs repo IDs.')

def test_validate_local_file_missing():
    from pathlib import Path
    try:
        _validate_local_file(Path('/nonexistent/file.txt'))
        assert False
    except FatTummyDatasetError as e:
        print(f'  Correct error for missing file: {e}')

def test_validate_local_file_bad_extension():
    from pathlib import Path
    with tempfile.NamedTemporaryFile(suffix='.parquet', delete=False) as f:
        tmp = f.name
    try:
        _validate_local_file(Path(tmp))
        assert False
    except FatTummyDatasetError as e:
        print(f'  Correct error for bad extension: {e}')
    finally:
        os.unlink(tmp)

def test_validate_hf_repo_id_valid():
    _validate_hf_repo_id('namespace/repo')   # no exception
    _validate_hf_repo_id('my-org/my_model.v2')
    print('  Valid repo IDs accepted.')

def test_validate_hf_repo_id_invalid():
    try:
        _validate_hf_repo_id('no-slash')
        assert False
    except FatTummyDatasetError as e:
        print(f'  Correct error for missing slash: {e}')

def test_parse_hf_source_simple():
    repo, config = _parse_hf_source('ns/repo')
    assert repo == 'ns/repo' and config is None
    print(f'  Parsed simple: repo={repo}, config={config}')

def test_parse_hf_source_with_config_colon():
    repo, config = _parse_hf_source('ns/repo:mycfg')
    assert repo == 'ns/repo' and config == 'mycfg'
    print(f'  Parsed with colon config: repo={repo}, config={config}')

def test_parse_hf_source_with_config_slash():
    repo, config = _parse_hf_source('ns/repo/subcfg')
    assert repo == 'ns/repo' and config == 'subcfg'
    print(f'  Parsed with slash config: repo={repo}, config={config}')

run_test('_split_sources string', test_split_sources_string)
run_test('_split_sources list', test_split_sources_list)
run_test('_split_sources None', test_split_sources_none)
run_test('_split_sources invalid type', test_split_sources_invalid_type)
run_test('_looks_like_path', test_looks_like_path_txt)
run_test('_validate_local_file missing', test_validate_local_file_missing)
run_test('_validate_local_file bad extension', test_validate_local_file_bad_extension)
run_test('_validate_hf_repo_id valid', test_validate_hf_repo_id_valid)
run_test('_validate_hf_repo_id invalid', test_validate_hf_repo_id_invalid)
run_test('_parse_hf_source simple', test_parse_hf_source_simple)
run_test('_parse_hf_source colon config', test_parse_hf_source_with_config_colon)
run_test('_parse_hf_source slash config', test_parse_hf_source_with_config_slash)

---
## 9. Trainer (CPU, no HF)

In [ ]:
if _HAS_TORCH:
    from FatTummy.tuning.trainer import FatTummyTrainer, _encode_text, _collate_batch
    from FatTummy.models.mooe import MOOE, MOOEConfig
    import torch

    def _make_tiny_model():
        cfg = MOOEConfig(hidden_size=32, intermediate_size=64, num_layers=1, num_experts=2, top_k=1, vocab_size=64)
        return MOOE(cfg)

    def test_encode_text_no_tokenizer():
        result = _encode_text('hello', tokenizer=None, max_length=8)
        assert 'input_ids' in result and 'labels' in result
        assert len(result['input_ids']) <= 8
        assert result['input_ids'] == result['labels']
        print(f'  _encode_text (no tokenizer): {result}')

    def test_encode_text_short_padded():
        # Single char → padded to length 2
        result = _encode_text('A', tokenizer=None, max_length=8)
        assert len(result['input_ids']) >= 2
        print(f'  _encode_text short padded: {result}')

    def test_collate_batch():
        examples = [
            {'input_ids': [1,2,3], 'labels': [1,2,3]},
            {'input_ids': [4,5], 'labels': [4,5]},
        ]
        batch = _collate_batch(examples)
        assert batch['input_ids'].shape == (2, 3), f'bad shape: {batch["input_ids"].shape}'
        assert batch['labels'][1, -1].item() == -100, 'padding label should be -100'
        print(f'  _collate_batch: input_ids shape={batch["input_ids"].shape}, label pad=-100 OK')

    def test_trainer_one_epoch_native():
        model = _make_tiny_model()
        dataset = ['hello world', 'foo bar baz', 'test training data']
        trainer = FatTummyTrainer(
            model, dataset, tokenizer=None,
            epochs=1, batch_size=2, max_length=16,
            output_dir='test_fattummy_ckpt'
        )
        trainer.finetune()
        print('  Trainer 1-epoch native run completed.')

    def test_trainer_iter_texts_list_of_dicts():
        model = _make_tiny_model()
        dataset = [{'text': 'hello'}, {'text': 'world'}, {'content': 'foo'}]
        trainer = FatTummyTrainer(model, dataset, tokenizer=None, epochs=1, max_length=16, output_dir='test_fattummy_ckpt2')
        trainer.finetune()
        print('  Trainer from list-of-dicts with text key OK.')

    def test_trainer_select_split():
        model = _make_tiny_model()
        dataset = {'train': ['sample a', 'sample b'], 'test': ['sample c']}
        trainer = FatTummyTrainer(model, dataset, epochs=1, max_length=16, output_dir='test_fattummy_ckpt3')
        split = trainer._select_split(dataset)
        assert split == ['sample a', 'sample b']
        print('  _select_split returns train split correctly.')

    def test_trainer_evaluate():
        model = _make_tiny_model()
        import torch
        dataset = ['hello world', 'testing evaluate']
        trainer = FatTummyTrainer(model, dataset, eval_dataset=dataset, epochs=1, max_length=16, output_dir='test_fattummy_ckpt4')
        loss = trainer._evaluate(torch.device('cpu'))
        assert loss is not None and loss > 0
        print(f'  _evaluate returned loss={loss:.4f}')

    run_test('_encode_text (no tokenizer)', test_encode_text_no_tokenizer)
    run_test('_encode_text short → padded', test_encode_text_short_padded)
    run_test('_collate_batch padding', test_collate_batch)
    run_test('Trainer 1-epoch native', test_trainer_one_epoch_native)
    run_test('Trainer from list-of-dicts', test_trainer_iter_texts_list_of_dicts)
    run_test('Trainer _select_split', test_trainer_select_split)
    run_test('Trainer _evaluate', test_trainer_evaluate)

    # Cleanup test checkpoint dirs
    import shutil
    for d in ['test_fattummy_ckpt', 'test_fattummy_ckpt2', 'test_fattummy_ckpt3', 'test_fattummy_ckpt4']:
        if os.path.exists(d):
            shutil.rmtree(d)
    print('Test checkpoint dirs cleaned up.')
else:
    print('Skipping Trainer tests (PyTorch not installed)')

---
## 10. Cloud Adapters (no real keys)

In [ ]:
from FatTummy.inference.cloud_adapters import get_cloud_adapter
from FatTummy.exceptions import FatTummyAuthenticationError, FatTummyUnsupportedBackendError

def test_get_cloud_adapter_unknown():
    try:
        get_cloud_adapter('fakeprovider', api_key='x')
        assert False
    except FatTummyUnsupportedBackendError as e:
        print(f'  Correct error for unknown cloud engine: {e}')

def test_cloud_adapter_empty_key():
    """BaseCloudAdapter should raise AuthError if key is empty."""
    # Test via a subclass that doesn't try to import the SDK
    from FatTummy.inference.cloud_adapters import BaseCloudAdapter
    from dataclasses import dataclass

    @dataclass
    class _DummyAdapter(BaseCloudAdapter):
        def generate(self, prompt): return ''

    try:
        _DummyAdapter(api_key='')  # should raise in __post_init__
        assert False
    except FatTummyAuthenticationError as e:
        print(f'  Correct AuthError for empty key: {e}')

run_test('get_cloud_adapter unknown', test_get_cloud_adapter_unknown)
run_test('BaseCloudAdapter empty key', test_cloud_adapter_empty_key)

---
## 11. Local Adapters

In [ ]:
from FatTummy.inference.local_adapters import get_local_adapter
from FatTummy.exceptions import FatTummyUnsupportedBackendError

def test_get_local_adapter_unknown():
    try:
        get_local_adapter('unknownengine', 'model')
        assert False
    except FatTummyUnsupportedBackendError as e:
        print(f'  Correct error for unknown local engine: {e}')

def test_hf_adapter_creation():
    """HuggingFaceAdapter should be created lazily (no model loading yet)."""
    from FatTummy.inference.local_adapters import HuggingFaceAdapter
    adapter = HuggingFaceAdapter('gpt2', token=None)
    assert adapter.model is None, 'model should be None before _load()'
    assert adapter.tokenizer is None, 'tokenizer should be None before _load()'
    print('  HuggingFaceAdapter created lazily — model/tokenizer are None.')

run_test('get_local_adapter unknown', test_get_local_adapter_unknown)
run_test('HuggingFaceAdapter lazy creation', test_hf_adapter_creation)

---
## 12. Interactive Wizard — Advanced Mode & Bug Fixes

In [ ]:
from FatTummy.interactive import (
    validate_config, execute_config, _resolve_api_provider,
    NATIVE_CHOICES, _collect_for_action, _run_advanced
)
from FatTummy.exceptions import FatTummyConfigurationError

def test_resolve_api_provider_aliases():
    assert _resolve_api_provider('claude') == 'anthropic'
    assert _resolve_api_provider('google') == 'gemini'
    assert _resolve_api_provider('OPENAI') == 'openai'
    assert _resolve_api_provider('unknown') == 'unknown'
    print('  _resolve_api_provider aliases correct.')

def test_validate_config_finetune_no_dataset():
    config = {'action': 'finetune', 'datasets_raw': '', 'engine': 'hf'}
    try:
        validate_config(config)
        assert False, 'should have raised'
    except ValueError as e:
        print(f'  Correct error for finetune with no dataset: {e}')

def test_validate_config_api_no_key():
    config = {'action': 'api', 'datasets_raw': '', 'engine': 'openai', 'api_key': ''}
    try:
        validate_config(config)
        assert False, 'should have raised'
    except ValueError as e:
        print(f'  Correct error for api with no key: {e}')

def test_validate_config_normalizes_native_type():
    """validate_config should set type = engine when engine is a native choice."""
    config = {'action': 'make', 'datasets_raw': '', 'engine': 'mooe', 'type': 'something_else'}
    validate_config(config)
    assert config['type'] == 'mooe', f'type should be normalized to mooe, got {config["type"]}'
    print(f'  validate_config normalized type to: {config["type"]}')

def test_advanced_mode_has_quantize_prompt():
    """BUG CHECK: Advanced mode must expose quantize prompt.
    This test verifies the fixed _run_advanced() collects quantize."""
    from FatTummy.interactive import _run_advanced
    import inspect
    src = inspect.getsource(_run_advanced)
    assert 'quantize' in src or 'Quantize' in src, \
        'Advanced mode must ask for quantization! BUG: Full control is missing quantize prompt.'
    print('  _run_advanced() contains quantize prompt — Advanced mode has full control.')

def test_advanced_mode_has_timeout_prompt():
    """BUG CHECK: Advanced mode must expose timeout prompt."""
    from FatTummy.interactive import _run_advanced
    import inspect
    src = inspect.getsource(_run_advanced)
    assert 'timeout' in src or 'Timeout' in src, \
        'Advanced mode must ask for timeout! BUG: Full control is missing timeout prompt.'
    print('  _run_advanced() contains timeout prompt — Advanced mode has full control.')

def test_run_action_no_unconditional_chat():
    """BUG CHECK: _run_action should NOT always call chat() after finetune."""
    from FatTummy.interactive import _run_action
    import inspect
    src = inspect.getsource(_run_action)
    # A fixed _run_action should guard chat() behind a condition
    # It should NOT have a bare engine.chat() without a condition check
    lines = [l.strip() for l in src.splitlines()]
    # Find 'engine.chat()' lines — they must be inside an if block, not bare at end
    chat_lines = [i for i, l in enumerate(lines) if 'engine.chat()' in l]
    bare_chat = [lines[i] for i in chat_lines if lines[i] == 'engine.chat()']
    assert not bare_chat, f'BUG: bare engine.chat() found unconditionally: {bare_chat}'
    print('  _run_action: engine.chat() is not called unconditionally — BUG fixed.')

def test_apply_config_engine_before_type():
    """BUG CHECK: _apply_config should call engine.engine() BEFORE engine.type()."""
    from FatTummy.interactive import _apply_config
    import inspect
    src = inspect.getsource(_apply_config)
    engine_pos = src.find('engine.engine(')
    type_pos = src.find('engine.type(')
    assert engine_pos < type_pos, \
        f'BUG: engine.engine() (pos {engine_pos}) must be called before engine.type() (pos {type_pos})'
    print('  _apply_config calls engine.engine() before engine.type() — correct order.')

run_test('_resolve_api_provider aliases', test_resolve_api_provider_aliases)
run_test('validate_config: finetune no dataset', test_validate_config_finetune_no_dataset)
run_test('validate_config: api no key', test_validate_config_api_no_key)
run_test('validate_config: normalizes native type', test_validate_config_normalizes_native_type)
run_test('[BUG] Adv mode: quantize prompt present', test_advanced_mode_has_quantize_prompt)
run_test('[BUG] Adv mode: timeout prompt present', test_advanced_mode_has_timeout_prompt)
run_test('[BUG] _run_action: no unconditional chat()', test_run_action_no_unconditional_chat)
run_test('[BUG] _apply_config: engine() before type()', test_apply_config_engine_before_type)

---
## 13. Engine.finetune() edge cases

In [ ]:
from FatTummy.engine import FatTummyEngine
from FatTummy.exceptions import FatTummyConfigurationError, FatTummyUnsupportedBackendError

def test_finetune_no_dataset_raises():
    e = FatTummyEngine()
    e.type('mooe')
    try:
        e.finetune()
        assert False, 'should have raised'
    except FatTummyConfigurationError as ex:
        print(f'  Correct error for finetune with no dataset: {ex}')

def test_finetune_cloud_engine_raises():
    e = FatTummyEngine()
    e._engine_name = 'openai'
    e._compiled = True
    e._data_sources = ['some_data']
    try:
        e.finetune()
        assert False, 'should have raised'
    except FatTummyUnsupportedBackendError as ex:
        print(f'  Correct error for finetune on cloud engine: {ex}')

if _HAS_TORCH:
    def test_finetune_native_model_runs():
        e = FatTummyEngine()
        e.type('mooe').modelbuild('tiny')
        e.data('hello world', 'foo bar', 'baz qux')
        e.finetune(epochs=1)
        import shutil
        if os.path.exists('fattummy_checkpoints'):
            shutil.rmtree('fattummy_checkpoints')
        print('  engine.finetune() on native mooe model ran 1 epoch.')

    run_test('finetune: no dataset raises', test_finetune_no_dataset_raises)
    run_test('finetune: cloud engine raises', test_finetune_cloud_engine_raises)
    run_test('finetune: native model 1 epoch', test_finetune_native_model_runs)
else:
    run_test('finetune: no dataset raises', test_finetune_no_dataset_raises)
    run_test('finetune: cloud engine raises', test_finetune_cloud_engine_raises)
    print('Skipping native finetune test (no torch)')

---
## 14. Global ft.* API

In [ ]:
import FatTummy as ft

def test_global_build_non_interactive():
    engine = ft.build(interactive=False)
    assert engine is not None
    print(f'  ft.build(interactive=False) returned: {type(engine).__name__}')

def test_global_modelbuild():
    ft.build(interactive=False)
    result = ft.modelbuild('small')
    assert result is not None
    print('  ft.modelbuild("small") OK')

def test_global_temp():
    ft.build(interactive=False)
    ft.temp(0.8)
    print('  ft.temp(0.8) OK')

def test_global_key():
    ft.build(interactive=False)
    ft.key('sk-test')
    print('  ft.key() OK')

def test_global_engine():
    ft.build(interactive=False)
    ft.engine('mooe')
    print('  ft.engine("mooe") OK')

def test_global_getattr_missing():
    try:
        _ = ft.nonexistent_attribute_xyz
        assert False
    except AttributeError as e:
        print(f'  Correct AttributeError for missing attr: {e}')

run_test('ft.build(interactive=False)', test_global_build_non_interactive)
run_test('ft.modelbuild()', test_global_modelbuild)
run_test('ft.temp()', test_global_temp)
run_test('ft.key()', test_global_key)
run_test('ft.engine()', test_global_engine)
run_test('ft.nonexistent → AttributeError', test_global_getattr_missing)

---
## 📊 Summary

In [ ]:
passed = sum(1 for _, ok, _ in _results if ok)
failed = sum(1 for _, ok, _ in _results if not ok)
total  = len(_results)

print(f'\n{"="*60}')
print(f'  TOTAL: {total}   PASSED: {passed}   FAILED: {failed}')
print(f'{"="*60}')

if failed:
    print('\nFailed tests:')
    for name, ok, err in _results:
        if not ok:
            print(f'  ❌  {name}: {err}')
else:
    print('\n🎉  All tests passed!')